In [ ]:
import ast
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import emoji
import pymysql
from scipy import stats
from sqlalchemy import create_engine
from matplotlib.patches import Patch
from tqdm.auto import tqdm
from IPython.display import display

warnings.filterwarnings('ignore')

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

np.random.seed(42)

print("라이브러리 로드 완료")

라이브러리 로드 완료!
한글 폰트 설정 완료!


# 1단계 : 리뷰 텍스트 기본 전처리

## 목적
무신사 리뷰 텍스트를 임베딩/토픽 모델링/감성분석에 사용할 수 있도록 노이즈를 제거하고, 분석 단계별로 사용할 컬럼을 생성한다.

## 입력 / 출력
- **입력**: `cleaned_reviews.csv` (685,292건)
- **출력**: `reviews_step1_cleaned.csv`

## 생성 컬럼 요약
| 컬럼 | 용도 | 처리 강도 |
|---|---|---|
| `리뷰내용` | 원본 보존 (수정 X) | - |
| `리뷰내용_clean` | 임베딩, 토픽 모델링용 | 약한 정제 |
| `리뷰내용_norm` | 중복 검출용 (비교 키) | 강한 정규화 |
| `laugh_count`, `cry_count`, `exclamation_count`, `question_count`, `emoji_count`, `has_repetition`, `text_length_orig` | 감정 강도 플래그 | 원본에서 추출 |
| `is_valid_for_topic` | 토픽 모델링 입력 여부 (한글 5자 이상) | - |

## 처리 절차
1. 데이터 로드
2. 노이즈 패턴 확인
3. 정제 함수 정의
4. 정제 함수 적용
5. 감정 강도 플래그 추출
6. 유효 리뷰 플래그 생성
7. 전후 비교 검증
8. 빈 리뷰 확인
9. 저장

## 1. 데이터 로드

원본 CSV를 불러오고, 리뷰 본문이 NaN인 행을 제거한다.

In [ ]:
cp = pd.read_csv('data/4m_cleaned_products.csv')
cr = pd.read_csv('data/4m_cleaned_reviews.csv')
df_v1 = cr.copy()

print(f'products: {len(cp):,}건')
print(f'reviews:  {len(cr):,}건')


products: 2,986건
reviews:  2,376건


In [13]:
df = cr.copy()

TEXT_COL = "리뷰내용"
# 리뷰 본문 NaN 제거
before = len(df)
df = df.dropna(subset=[TEXT_COL]).reset_index(drop=True)
print(f"NaN 제거: {before:,} → {len(df):,}건 (-{before-len(df):,})")

NaN 제거: 2,376 → 2,376건 (-0)


## 2. 노이즈 패턴 확인

정제 전, 데이터에 어떤 노이즈 패턴이 얼마나 분포하는지 확인하여 정제 대상을 파악한다.

In [14]:
# 노이즈 패턴 분포 확인
sample = df[TEXT_COL].astype(str).astype('object')

print("[노이즈 패턴 분포]")
print(f"엑셀 오류값(#NAME? 등): {sample.str.contains(r'#(?:NAME|N/A|VALUE|DIV/0|REF|NULL|NUM)', regex=True, na=False).sum():,}건")
print(f"HTML 태그(<br> 등):     {sample.str.contains(r'<[^>]+>', regex=True, na=False).sum():,}건")
print(f"HTML 엔티티(&nbsp;):    {sample.str.contains(r'&[a-z]+;', regex=True, na=False).sum():,}건")
print(f"URL:                  {sample.str.contains(r'https?://|www\.', regex=True, na=False).sum():,}건")
print(f"줄바꿈/탭:             {sample.str.contains(r'[\r\n\t]', regex=True, na=False).sum():,}건")
print(f"반복문자 4회+:          {sample.str.contains(r'(.)\1{3,}', regex=True, na=False).sum():,}건")
print(f"이모지 포함:           {sample.apply(lambda x: emoji.emoji_count(x) > 0).sum():,}건")
print(f"한글 없음:             {(~sample.str.contains(r'[가-힣]', regex=True, na=False)).sum():,}건")

[노이즈 패턴 분포]
엑셀 오류값(#NAME? 등): 0건
HTML 태그(<br> 등):     1건
HTML 엔티티(&nbsp;):    0건
URL:                  0건
줄바꿈/탭:             0건
반복문자 4회+:          22건
이모지 포함:           44건
한글 없음:             2건


## 3. 정제 함수 정의

목적이 다른 두 가지 정제 함수를 정의한다.

- **`clean_review_text`** : 약한 정제. 의미를 보존하면서 노이즈만 제거. 임베딩과 토픽 모델링에 사용.
- **`normalize_review_text`** : 강한 정규화. 중복 검출 시 동일 리뷰를 같은 키로 매핑하기 위한 용도.

> **`리뷰내용_norm`에서 `%`, `/` 같은 기호를 모두 제거하는 이유**
> 이 컬럼은 사람이 읽거나 분석에 쓰는 텍스트가 아니라 "이 두 리뷰가 같은가?"를 비교하는 **지문(fingerprint)** 역할이다. 같은 내용을 약간 다른 형식으로 쓴 리뷰들이 동일하게 매칭되어야 중복 검출이 정확해진다.
> 의미 정보는 `리뷰내용_clean`에 그대로 보존되어 있다.

In [ ]:
def clean_review_text(text):
    """
    약한 정제 (임베딩, 토픽 모델링용).
    의미는 보존하면서 노이즈만 제거.
    """
    text = str(text)
    
    # (1) 엑셀 오류값
    text = re.sub(r'#(?:NAME|N/A|VALUE|DIV/0|REF|NULL|NUM)[!?]?', ' ', text)
    
    # (2) URL
    text = re.sub(r'http\S+|www\.\S+', ' ', text)
    
    # (3) HTML 태그 + 엔티티
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'&[a-zA-Z]+;', ' ', text)
    
    # (4) 줄바꿈/탭 → 공백
    text = re.sub(r'[\r\n\t]+', ' ', text)
    
    # (5) 이모지 제거 (감성 신호는 emoji_count 플래그로 별도 보존)
    text = emoji.replace_emoji(text, replace=' ')
    
    # (6) 반복문자 정규화 (약하게: 4회+ → 3회로 압축)
    text = re.sub(r'([ㄱ-ㅎㅏ-ㅣ])\1{3,}', r'\1\1\1', text)   # 자모
    text = re.sub(r'(.)\1{3,}', r'\1\1\1', text)            # 일반 문자
    
    # (7) 연속 공백 정리
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text


# 셀 3의 normalize_review_text 함수 교체
def normalize_review_text(text):
    """
    강한 정규화 (중복 검출 전용).
    자모만 압축하고 숫자/영문/한글 완성형은 보존.
    """
    text = str(text)
    
    # jaccard -> 코드상 대소문자 그대로 비교
    # 일관된 처리를 위해 모두 소문자로 변환
    text = text.lower()

    # 기본 노이즈 제거
    text = re.sub(r'#(?:NAME|N/A|VALUE|DIV/0|REF|NULL|NUM)[!?]?', '', text)
    text = re.sub(r'http\S+|www\.\S+', '', text)
    text = re.sub(r'<[^>]+>', '', text)
    text = re.sub(r'&[a-zA-Z]+;', '', text)
    
    # 이모지 완전 제거
    text = emoji.replace_emoji(text, replace='')
    
    # 수정: 자모만 1개로 압축 (숫자/영문/한글 완성형은 보존)
    text = re.sub(r'([ㄱ-ㅎㅏ-ㅣ])\1+', r'\1', text)
    
    # 한글/영문/숫자만 남기고 모두 제거 (공백, 문장부호 포함)
    text = re.sub(r'[^가-힣a-zA-Z0-9]', '', text)
    
    return text

print("정제 함수 정의 완료")


정제 함수 정의 완료


`%나 /같은 기호를 빼는 이유!` : 리뷰내용_norm은 중복 검출 전용 컬럼이기 때문.

## 4. 정제 함수 적용

두 함수를 데이터프레임 전체에 적용해 `리뷰내용_clean`, `리뷰내용_norm` 컬럼을 생성한다.

In [16]:
tqdm.pandas()

df['리뷰내용_clean'] = df[TEXT_COL].progress_apply(clean_review_text)
df['리뷰내용_norm']  = df[TEXT_COL].progress_apply(normalize_review_text)

print("정제 완료")
print(f"  리뷰내용_clean: 약한 정제 (임베딩/토픽 모델링용)")
print(f"  리뷰내용_norm : 강한 정규화 (중복 검출용)")

  0%|          | 0/2376 [00:00<?, ?it/s]

  0%|          | 0/2376 [00:00<?, ?it/s]

정제 완료
  리뷰내용_clean: 약한 정제 (임베딩/토픽 모델링용)
  리뷰내용_norm : 강한 정규화 (중복 검출용)


In [17]:
print(f'df 행 수: {len(df)}')
print(f'TEXT_COL: {TEXT_COL}')
print(f'df 컬럼: {df.columns.tolist()}')

df 행 수: 2376
TEXT_COL: 리뷰내용
df 컬럼: ['goodsNo', '리뷰번호', '리뷰타입', '작성자', '리뷰내용', '평점', '체험단', '구매옵션', '키', '몸무게', '성별', '작성일', '만족도', '사진유무', '도움돼요', '구매사이즈', '구매상세', '연도', '월', '일', '요일', '평점_raw', '만족도_응답여부', '사이즈', '화면대비색감', '퀄리티', '구김', '색감', '신축성', '두께감', '보온성', '퀄리티_점수', '보온성_점수', '신축성_점수', '두께감_점수', '구김_점수', '사이즈_편차', '사이즈_편차절대', '화면대비색감_편차', '화면대비색감_편차절대', '색감_편차', '색감_편차절대', '노출일수', '일평균_도움돼요지수', '도움여부', '리뷰내용_clean', '리뷰내용_norm']


## 5. 감정 강도 플래그 추출 (원본 기준)

정제 과정에서 압축되는 감정 신호(`ㅋㅋㅋㅋ`, `!!!`, 이모지 등)의 강도를 별도 컬럼으로 보존한다.
**원본 텍스트** 기준으로 추출해야 정확한 강도가 측정된다 (정제 후엔 이미 압축되어 있음).

| 플래그 | 의미 |
|---|---|
| `laugh_count` | ㅋ/ㅎ 연속 최대 길이 |
| `cry_count` | ㅠ/ㅜ 연속 최대 길이 |
| `exclamation_count` | `!` 개수 |
| `question_count` | `?` 개수 |
| `emoji_count` | 이모지 개수 |
| `has_repetition` | 반복문자 존재 여부 (bool) |
| `text_length_orig` | 원본 길이 |

In [18]:
def extract_flags(text):
    """원본 텍스트에서 감정 강도/통계 플래그 추출."""
    text = str(text)
    
    # ㅋ/ㅎ 연속 최대 길이
    laugh_matches = re.findall(r'[ㅋㅎ]+', text)
    laugh_count = max((len(m) for m in laugh_matches), default=0)
    
    # ㅠ/ㅜ 연속 최대 길이
    cry_matches = re.findall(r'[ㅠㅜ]+', text)
    cry_count = max((len(m) for m in cry_matches), default=0)
    
    return pd.Series({
        'laugh_count':       laugh_count,
        'cry_count':         cry_count,
        'exclamation_count': text.count('!'),
        'question_count':    text.count('?'),
        'emoji_count':       emoji.emoji_count(text),
        'has_repetition':    bool(re.search(r'(.)\1{2,}', text)),
        'text_length_orig':  len(text),
    })

# 원본 텍스트 기준으로 플래그 추출
flags = df[TEXT_COL].progress_apply(extract_flags)
df = pd.concat([df, flags], axis=1)

print("플래그 추출 완료")
print(df[['laugh_count','cry_count','exclamation_count',
          'question_count','emoji_count','has_repetition',
          'text_length_orig']].describe().round(2))

  0%|          | 0/2376 [00:00<?, ?it/s]

플래그 추출 완료
       laugh_count  cry_count  exclamation_count  question_count  emoji_count  \
count      2376.00    2376.00            2376.00         2376.00      2376.00   
mean          0.08       0.03               0.30            0.03         0.03   
std           0.38       0.24               0.84            0.19         0.22   
min           0.00       0.00               0.00            0.00         0.00   
25%           0.00       0.00               0.00            0.00         0.00   
50%           0.00       0.00               0.00            0.00         0.00   
75%           0.00       0.00               0.00            0.00         0.00   
max           3.00       3.00              12.00            2.00         4.00   

       text_length_orig  
count           2376.00  
mean              55.05  
std               58.94  
min               20.00  
25%               30.00  
50%               37.00  
75%               53.00  
max              635.00  


## 6. 유효 리뷰 플래그

한글 5자 미만 리뷰는 토픽 모델링에 의미가 없어 제외 대상으로 표시한다.
실제 행은 삭제하지 않고 `is_valid_for_topic` 플래그로만 구분하여, 이후 단계에서 선택적으로 필터링할 수 있게 한다.

In [19]:
df['한글_글자수']  = df['리뷰내용_clean'].str.count(r'[가-힣]')
df['전체_글자수']  = df['리뷰내용_clean'].str.len()
df['is_valid_for_topic'] = df['한글_글자수'] >= 5

print(f"[유효 리뷰 분포]")
print(f"  유효 (한글 5자 이상): {df['is_valid_for_topic'].sum():,}건 "
      f"({df['is_valid_for_topic'].mean()*100:.2f}%)")
print(f"  무효 (한글 5자 미만): {(~df['is_valid_for_topic']).sum():,}건")

print("\n[무효로 분류된 리뷰 샘플]")
display(df.loc[~df['is_valid_for_topic'], 
               ['리뷰내용', '리뷰내용_clean', '한글_글자수']].head(10))

[유효 리뷰 분포]
  유효 (한글 5자 이상): 2,374건 (99.92%)
  무효 (한글 5자 미만): 2건

[무효로 분류된 리뷰 샘플]


,리뷰내용,리뷰내용_clean,한글_글자수
1204,I'm really enjoying wearing it. It's very comf...,I'm really enjoying wearing it. It's very comf...,0
1457,very very very good good good,very very very good good good,0


## 7. 전후 비교 검증

정제가 의도대로 동작했는지 샘플로 확인한다. 원본과 두 정제 컬럼을 나란히 비교한다.

In [20]:
# 정제 전후 비교
print("[정제 전후 비교 샘플]")
sample_idx = df[df['리뷰내용'] != df['리뷰내용_clean']].sample(10, random_state=42).index
for i in sample_idx:
    print(f"\n리뷰번호: {df.loc[i, '리뷰번호']}")
    print(f"  원본       : {repr(df.loc[i, '리뷰내용'])[:100]}")
    print(f"  clean      : {repr(df.loc[i, '리뷰내용_clean'])[:100]}")
    print(f"  norm(중복용): {repr(df.loc[i, '리뷰내용_norm'])[:100]}")

print("\n[글자 수 통계]")
print(df[['전체_글자수', '한글_글자수']].describe())

changed = (df['리뷰내용'].astype(str) != df['리뷰내용_clean']).sum()
print(f"\n정제로 인해 변경된 리뷰: {changed:,}건 ({changed/len(df)*100:.1f}%)")

[정제 전후 비교 샘플]

리뷰번호: 83721968
  원본       : '마감이 살짝 아쉽긴 한데 가격 생각하면 뭐..  이 가격에 이런 원담감과 넥라인, 색감 찾기 힘듭니다'
  clean      : '마감이 살짝 아쉽긴 한데 가격 생각하면 뭐.. 이 가격에 이런 원담감과 넥라인, 색감 찾기 힘듭니다'
  norm(중복용): '마감이살짝아쉽긴한데가격생각하면뭐이가격에이런원담감과넥라인색감찾기힘듭니다'

리뷰번호: 83761485
  원본       : '기장감도 좋고  핏도 예뻐요 딱 좋은것 같아요 편하게 입기'
  clean      : '기장감도 좋고 핏도 예뻐요 딱 좋은것 같아요 편하게 입기'
  norm(중복용): '기장감도좋고핏도예뻐요딱좋은것같아요편하게입기'

리뷰번호: 83521800
  원본       : '생각보다 두껍다는 평이 많았는데 완전 한여름용 우비같은 얇음은 아니고 적당히 얇으면서 보온성 있는 고급스러운 재질이에요 (나이키 바람막이 생가하시면 좋을듯) 그리고 사이즈 클까봐
  clean      : '생각보다 두껍다는 평이 많았는데 완전 한여름용 우비같은 얇음은 아니고 적당히 얇으면서 보온성 있는 고급스러운 재질이에요 (나이키 바람막이 생가하시면 좋을듯) 그리고 사이즈 클까봐
  norm(중복용): '생각보다두껍다는평이많았는데완전한여름용우비같은얇음은아니고적당히얇으면서보온성있는고급스러운재질이에요나이키바람막이생가하시면좋을듯그리고사이즈클까봐걱정했는데제키랑비슷하시면라지추천드려요저는항

리뷰번호: 83471444
  원본       : '이뻐요.  사이즈도 딱 좋아요.  사진보다  실물이 더 좋네요'
  clean      : '이뻐요. 사이즈도 딱 좋아요. 사진보다 실물이 더 좋네요'
  norm(중복용): '이뻐요사이즈도딱좋아요사진보다실물이더좋네요'

리뷰번호: 83465315
  원본       : '생각했던 것보단 얇은 감이 있지만,  스타일리쉬하고 지금같은 계절에 후드티 입고 입기 좋네요. 2온스라 겨울용은 아닙니다.'
  clea

## 8. 빈 리뷰 확인

정제 후 텍스트가 비어버린 리뷰를 확인한다. 일반적으로 URL만 있던 리뷰나 이모지로만 구성된 리뷰가 해당된다.
이런 행도 `is_valid_for_topic = False`로 자동 분류되어 토픽 모델링에서는 제외된다.

In [21]:
empty_after = df[df['전체_글자수'] == 0]
print(f"정제 후 빈 리뷰: {len(empty_after):,}건")
print("\n[원본 확인]")
display(empty_after[['리뷰번호', '리뷰내용']].head(20))

정제 후 빈 리뷰: 0건

[원본 확인]


,리뷰번호,리뷰내용


## 9. 저장

전처리 결과를 `reviews_step1_cleaned.csv`로 저장한다. 다음 단계(중복 처리)에서 이 파일을 입력으로 사용한다.

In [22]:
output_path = "./data/4m_reviews_step1_cleaned.csv"
df.to_csv(output_path, index=False)
print(f"저장 완료: {output_path}")
print(f"\n[저장된 컬럼 요약]")
print(f"  텍스트 컬럼: 리뷰내용(원본), 리뷰내용_clean, 리뷰내용_norm")
print(f"  플래그    : laugh_count, cry_count, exclamation_count,")
print(f"             question_count, emoji_count, has_repetition, text_length_orig")
print(f"  유효성    : 한글_글자수, 전체_글자수, is_valid_for_topic")
print(f"\n전체 행 수: {len(df):,}건")

저장 완료: ./data/4m_reviews_step1_cleaned.csv

[저장된 컬럼 요약]
  텍스트 컬럼: 리뷰내용(원본), 리뷰내용_clean, 리뷰내용_norm
  플래그    : laugh_count, cry_count, exclamation_count,
             question_count, emoji_count, has_repetition, text_length_orig
  유효성    : 한글_글자수, 전체_글자수, is_valid_for_topic

전체 행 수: 2,376건
